In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import *

In [0]:
accounts_df = spark.read.table("bankaml.silver.accounts")
transactions_df = spark.read.table("bankaml.silver.transactions")

In [0]:
accounts_txns_df = (
    accounts_df.alias("a").join(
        transactions_df.alias("t"),
        (col("a.account_id") == col("t.account_id"))
        & (to_date(col("t.txn_ts")) >= col("a.effective_start_date"))
        & (to_date(col("t.txn_ts")) < col("a.effective_end_date")),
        "left"
    )
    .select(
        col("a.account_id").alias("account_id"),
        col("a.customer_id").alias("customer_id"),
        col("a.branch_id").alias("branch_id"),
        col("a.status").alias("account_status"),
        to_date(col("t.txn_ts")).alias("txn_date"),
        col("t.txn_id").alias("txn_id"),
        col("t.amount_usd").alias("amount_usd"),
        col("t.txn_type").alias("txn_type")
    )
)

IN_TXN_TYPES = ['wire_in', 'deposit', 'transfer_in',]
OUT_TXN_TYPES = ['wire_out', 'withdrawal',  'transfer_out']

accounts_summary_df = (
    accounts_txns_df.groupBy(col("account_id"), col("txn_date"))
    .agg(
        countDistinct(col("txn_id")).cast(IntegerType()).alias("txn_count"),
        sum(when(col("txn_type").isin(IN_TXN_TYPES), col("amount_usd"))).cast(DecimalType(18,2)).alias("total_deposit_usd"),
        sum(when(col("txn_type").isin(OUT_TXN_TYPES), col("amount_usd"))).cast(DecimalType(18,2)).alias("total_withdrawal_usd"),
        (coalesce(col("total_deposit_usd"), lit(0)).cast(DecimalType(18,2)) - coalesce(col("total_withdrawal_usd"), lit(0))).alias("net_flow_usd")
    )
)

accounts_summary_df = (
    accounts_txns_df.alias("at").join(
        accounts_summary_df.alias("as"),
        col("at.account_id") == col("as.account_id"),
        "left"
    )
    .withColumn(
        "status_activity_mismatch", 
        when((col("as.txn_count")>0) & 
                (col("at.account_status").isin(["dormant", "closed "])), 
            lit("Y")
        )
        .otherwise(lit("N"))
    )
    .withColumn(
        "composite_key",
        concat(col("at.account_id"), lit("_"), coalesce(col("at.txn_date").cast(StringType()), lit("")))
    )
    .select(
        col("at.account_id").alias("account_id"),
        col("at.customer_id").alias("customer_id"),
        col("at.branch_id").alias("branch_id"),
        col("at.txn_date").alias("txn_date"),
        col("as.txn_count").alias("txn_count"),
        col("as.total_deposit_usd").alias("total_deposit_usd"),
        col("as.total_withdrawal_usd").alias("total_withdrawal_usd"),
        col("as.net_flow_usd").alias("net_flow_usd"),
        col("status_activity_mismatch"),
        col("composite_key")
    )
)

In [0]:
%sql
create table if not exists bankaml.gold.account_daily_txn_summary
(
    account_id string,
    customer_id string, 
    branch_id string, 
    txn_date date, 
    txn_count int, 
    total_deposit_usd decimal(18,2), 
    total_withdrawal_usd decimal(18,2), 
    net_flow_usd decimal(18,2), 
    status_activity_mismatch string, 
    composite_key string
)
using delta;

In [0]:
accounts_summary_table = DeltaTable.forName(spark, "bankaml.gold.account_daily_txn_summary")
accounts_summary_df = accounts_summary_df.select(*accounts_summary_table.toDF().columns)

(
    accounts_summary_table.alias("t").merge(
        accounts_summary_df.alias("s"),
        "t.composite_key=s.composite_key"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)